# **2.1 Data Preparation**

The objective of this section is to transform the raw monthly trading data into a format that can be used efficiently for model fitting and backtesting.

The raw bin files are stored in long format, with one row per stock, date and intraday time bin. For price impact modelling, it is more convenient to work with matrices where each row corresponds to one stock-day and each column corresponds to one intraday time bin.

The baseline project setup uses one month as the in-sample training period and the following month as the out-of-sample testing period. The same set of 20 stocks is used in both periods.


Let

$$
q_{i,d,t}
$$

denote the signed traded volume of stock \(i\) on date \(d\) during intraday bin \(t\), and let

$$
P_{i,d,t}
$$

denote the mid price at the end of the same bin.

The aim is to construct two matrices:

$$
Q_{(i,d),t} = q_{i,d,t},
$$

and

$$
P_{(i,d),t} = P_{i,d,t}.
$$

Here, the row index \((i,d)\) represents one stock-day, while the columns represent intraday time bins.

We use January 2019 as the in-sample period and February 2019 as the out-of-sample period.

The in-sample data is used for stock selection and model fitting. The out-of-sample data is only used later to evaluate the fitted model.

This avoids look-ahead bias: the stock universe is selected using only the training month, not the testing month.

The 20-stock universe is selected by liquidity. For each stock \(i\), we compute total absolute traded volume in the training month:

$$
V_i = \sum_{d \in \text{train}} \sum_t |q_{i,d,t}|.
$$

The selected universe is then

$$
\mathcal{S}_{20}
=
\text{top 20 stocks ranked by } V_i.
$$

The same stock set $\mathcal{S}_{20}$ is then used for both the training month and the testing month.

The bin files contain both signed volume and mid prices.

For signed trading volume, we use the column `trade`:

$$
Q_{(i,d),t} = \text{trade}_{i,d,t}.
$$

For prices, we use the column `midEnd`:

$$
P_{(i,d),t} = \text{midEnd}_{i,d,t}.
$$

We convert the raw long data into wide stock-day matrices:

- rows: `(stock, date)`;
- columns: `time`;
- values: either `trade` or `midEnd`.

Missing values are treated differently for trades and prices.

For traded volume, missing values are filled with zero:

$$
q_{i,d,t} = 0
$$

when there is no recorded trade in that bin.

For prices, missing values are forward-filled and backward-filled across time because the absence of a new price observation does not mean that the price is zero. The last available mid price is carried forward.

The output of the data preparation step is:

$$
Q^{\text{train}}, \quad P^{\text{train}}, \quad Q^{\text{test}}, \quad P^{\text{test}}.
$$

These matrices are saved as intermediate results and will be used in the next section to fit price impact models.

This completes the baseline data preparation step.

The bin files are used for the baseline impact-fitting pipeline because they are already sampled on a regular 10-second grid and contain both signed traded volume and mid prices.

The fill files are not merged at this stage. They are event-level execution data with irregular timestamps, so directly merging them with the bin files would create unnecessary timestamp-alignment complications. They are kept separate and can be used later for markouts or execution-level diagnostics.

In [5]:
import os
import importlib
import src.data_prep

importlib.reload(src.data_prep)

from src.data_prep import *

data_dir = "data/"

bin_sample_path = f"{data_dir}binSamples/"
fill_sample_path = f"{data_dir}fillSamples/"

print(os.listdir(bin_sample_path))
print(os.listdir(fill_sample_path))

['bin201901.csv', 'bin201902.csv', 'bin201903.csv', 'bin201904.csv', 'bin201905.csv', 'bin201906.csv', 'bin201907.csv', 'bin201908.csv', 'bin201909.csv', 'bin201910.csv', 'bin201911.csv', 'bin201912.csv']


FileNotFoundError: [WinError 3] Le chemin d’accès spécifié est introuvable: 'data/fillSamples/'

In [2]:
import os
# Reading data
# January 2019 = in-sample
# February 2019 = out-of-sample

year = 2019
train_month = 1
test_month = 2

train_bin_df = load_bin_month(bin_sample_path, year, train_month)
test_bin_df = load_bin_month(bin_sample_path, year, test_month)


FileNotFoundError: [Errno 2] No such file or directory: 'data/binSamples/bin201901.csv'

In [48]:
# Choosing the 20 most liquid stocks
stocks_20 = select_top_liquid_stocks(train_bin_df, n_stocks=20)

In [49]:
# Saving the stocks
result_path = "data/"
os.makedirs(result_path, exist_ok=True)

pd.DataFrame({"stock": stocks_20}).to_csv(
    result_path + "stocks_20_201901.csv",
    index=False
)

In [50]:
# Filtering train and test to the same 20 stocks
train_bin_df = train_bin_df.loc[train_bin_df["stock"].isin(stocks_20)].copy()
test_bin_df = test_bin_df.loc[test_bin_df["stock"].isin(stocks_20)].copy()


train_bin_df = train_bin_df.sort_values(["stock", "date", "time"])
test_bin_df = test_bin_df.sort_values(["stock", "date", "time"])

In [51]:
# Build stock-date x time matrices
train_traded_volume_df = make_panel(train_bin_df, "trade", "zero")
test_traded_volume_df = make_panel(test_bin_df, "trade", "zero")
train_px_df = make_panel(train_bin_df, "midEnd", "price")
test_px_df = make_panel(test_bin_df, "midEnd", "price")

### **Data Checks**

In [52]:
# Align train and test time columns
(
    train_traded_volume_df,
    test_traded_volume_df,
    train_px_df,
    test_px_df
) = align_intraday_columns(
    train_traded_volume_df,
    test_traded_volume_df,
    train_px_df,
    test_px_df
)

In [53]:
train_traded_volume_df.head()

time              09:30:00  09:30:10  09:30:20  09:30:30  09:30:40  09:30:50  \
stock date                                                                     
AAL   2019-01-02      34.0       0.0      -8.0    -100.0     200.0    -857.0   
      2019-01-03       0.0      71.0    -479.0   -2244.0    1751.0     179.0   
      2019-01-04     885.0       3.0     679.0    2503.0     796.0   -3489.0   
      2019-01-07  -20132.0  -52657.0     515.0    1493.0    1475.0     157.0   
      2019-01-08    -974.0    2517.0   -1072.0      -1.0       0.0       0.0   

time              09:31:00  09:31:10  09:31:20  09:31:30  ...  15:58:30  \
stock date                                                ...             
AAL   2019-01-02     482.0       0.0    -210.0    -703.0  ...   -2062.0   
      2019-01-03   -3782.0     391.0     128.0   -6795.0  ...   16477.0   
      2019-01-04     -82.0       0.0     214.0    -239.0  ...    2431.0   
      2019-01-07     477.0     520.0     100.0    -502.0  ...    2097.0   
      2019-01-08    -458.0    -369.0       0.0       0.0  ...    -600.0   

time              15:58:40  15:58:50  15:59:00  15:59:10  15:59:20  15:59:30  \
stock date                                                                     
AAL   2019-01-02   -7739.0   -2160.0   -1213.0     275.0     887.0    3159.0   
      2019-01-03   10739.0   12621.0    2450.0  -30309.0   -5818.0   -4900.0   
      2019-01-04    1599.0    2055.0   -5121.0    4535.0   -4370.0   -1779.0   
      2019-01-07    1986.0   -1915.0     -26.0  -28476.0   -6700.0   -6031.0   
      2019-01-08  -15813.0   -1578.0    2172.0    2827.0  -17232.0  -15945.0   

time              15:59:40  15:59:50  16:00:00  
stock date                                      
AAL   2019-01-02   -1034.0    1615.0       0.0  
      2019-01-03  -13268.0   28753.0       0.0  
      2019-01-04   -3282.0   23432.0       0.0  
      2019-01-07  -19626.0   11838.0       0.0  
      2019-01-08  -23926.0     714.0       0.0  

[5 rows x 2341 columns]

### Stock-day information and training scaling variables

We first compute two quantities for each stock-day in the training sample.

For each stock $i$, date $d$, and intraday time bin $t$, let $q_{i,d,t}$ denote signed traded volume and let $P_{i,d,t}$ denote the mid price at the end of the bin.

The total absolute traded volume for stock $i$ on date $d$ is

$$
\text{volume}_{i,d}
=
\sum_t |q_{i,d,t}|.
$$

The intraday return is

$$
r_{i,d,t}
=
\frac{P_{i,d,t}}{P_{i,d,t-1}} - 1.
$$

The intraday volatility for stock $i$ on date $d$ is

$$
\text{px\_vol}_{i,d}
=
\operatorname{std}_t(r_{i,d,t}).
$$

Therefore, `train_stock_info_df` contains one row per stock-day:

$$
(i,d,\text{px\_vol}_{i,d},\text{volume}_{i,d}).
$$

This table keeps the daily information before averaging.

For the baseline model, we then compute one fixed scaling value per stock using only the training month. For each stock $i$, we average across training dates:

$$
\sigma_i
=
\frac{1}{N_i}
\sum_{d \in \text{train}}
\text{px\_vol}_{i,d},
$$

and

$$
ADV_i
=
\frac{1}{N_i}
\sum_{d \in \text{train}}
\text{volume}_{i,d}.
$$

Here, $N_i$ is the number of training days available for stock $i$.

The resulting `scaling_df` contains one row per stock:

$$
(i,\sigma_i,ADV_i).
$$

These training-only scaling variables will later be used to normalize price changes and traded volumes across stocks. We only use the training month to avoid look-ahead bias.

In [54]:
# Compute stock-day information 

train_stock_info_df = pd.DataFrame({
    "px_vol": train_px_df.pct_change(1, axis="columns").std(axis="columns"),
    "volume": train_traded_volume_df.abs().sum(axis="columns"),
}).reset_index()

scaling_df = (
    train_stock_info_df
    .groupby("stock")[["px_vol", "volume"]]
    .mean()
    .rename(columns={"px_vol": "sigma", "volume": "ADV"})
    .reset_index()
)

In [55]:
scaling_df.head()

,stock,sigma,ADV
0,AAL,0.000528,1.236722e+06
1,AAPL,0.000288,2.968463e+06
2,ABBV,0.000342,4.763432e+05
3,ABT,0.000283,5.000601e+05
4,ADBE,0.000327,4.143660e+05


### **Data Checks**

In [56]:
# 8. Diagnostics

print("Selected stocks:")
print(stocks_20)

print("\nShapes:")
print("train_traded_volume_df:", train_traded_volume_df.shape)
print("test_traded_volume_df :", test_traded_volume_df.shape)
print("train_px_df           :", train_px_df.shape)
print("test_px_df            :", test_px_df.shape)
print("train_stock_info_df   :", train_stock_info_df.shape)
print("scaling_df            :", scaling_df.shape)

print("\nNumber of stocks:")
print("Train volume:", train_traded_volume_df.index.get_level_values("stock").nunique())
print("Test volume :", test_traded_volume_df.index.get_level_values("stock").nunique())
print("Scaling     :", scaling_df["stock"].nunique())
print("Selected    :", len(stocks_20))

print("\nTime-column alignment:")
print("Train volume vs train price:", train_traded_volume_df.columns.equals(train_px_df.columns))
print("Test volume vs test price  :", test_traded_volume_df.columns.equals(test_px_df.columns))
print("Train vs test volume       :", train_traded_volume_df.columns.equals(test_traded_volume_df.columns))
print("Train vs test price        :", train_px_df.columns.equals(test_px_df.columns))

print("\nMissing values:")
print("train_traded_volume_df:", train_traded_volume_df.isna().sum().sum())
print("test_traded_volume_df :", test_traded_volume_df.isna().sum().sum())
print("train_px_df           :", train_px_df.isna().sum().sum())
print("test_px_df            :", test_px_df.isna().sum().sum())
print("train_stock_info_df   :", train_stock_info_df.isna().sum().sum())
print("scaling_df            :", scaling_df.isna().sum().sum())

print("\nTotal absolute traded volume:")
print("Train:", train_traded_volume_df.abs().sum().sum())
print("Test :", test_traded_volume_df.abs().sum().sum())

print("\nTraining stock-day information:")

print("\nTraining scaling variables:")

Selected stocks:
['AMD', 'AAPL', 'AMAT', 'AAL', 'AMZN', 'AMGN', 'ABT', 'ABBV', 'APA', 'ADI', 'ADBE', 'APC', 'AES', 'AIG', 'AFL', 'ADP', 'AEP', 'ALXN', 'ADM', 'ALGN']

Shapes:
train_traded_volume_df: (420, 2341)
test_traded_volume_df : (380, 2341)
train_px_df           : (420, 2341)
test_px_df            : (380, 2341)
train_stock_info_df   : (420, 4)
scaling_df            : (20, 3)

Number of stocks:
Train volume: 20
Test volume : 20
Scaling     : 20
Selected    : 20

Time-column alignment:
Train volume vs train price: True
Test volume vs test price  : True
Train vs test volume       : True
Train vs test price        : True

Missing values:
train_traded_volume_df: 0
test_traded_volume_df : 0
train_px_df           : 0
test_px_df            : 0
train_stock_info_df   : 0
scaling_df            : 0

Total absolute traded volume:
Train: 442042694.0
Test : 325471405.0

Training stock-day information:

Training scaling variables:


## **Saving files**

In [57]:
# Saving prepared panels

train_traded_volume_df.reset_index().to_csv(
    result_path + "train_traded_volume_201901_20.csv",
    index=False
)

train_px_df.reset_index().to_csv(
    result_path + "train_px_201901_20.csv",
    index=False
)

test_traded_volume_df.reset_index().to_csv(
    result_path + "test_traded_volume_201902_20.csv",
    index=False
)

test_px_df.reset_index().to_csv(
    result_path + "test_px_201902_20.csv",
    index=False
)

In [58]:
# Saving stock-day information

train_stock_info_df.to_csv(
    result_path + "train_stock_info_201901_20.csv",
    index=False
)

scaling_df.to_csv(
    result_path + "scaling_201901_20.csv",
    index=False
)

In [59]:
# Saving selected stock universe

pd.DataFrame({"stock": stocks_20}).to_csv(
    result_path + "stocks_20_201901.csv",
    index=False
)

In [60]:
scaling_df.head()

,stock,sigma,ADV
0,AAL,0.000528,1.236722e+06
1,AAPL,0.000288,2.968463e+06
2,ABBV,0.000342,4.763432e+05
3,ABT,0.000283,5.000601e+05
4,ADBE,0.000327,4.143660e+05
